# Phase 7: Decision Support, Caseload Triage, and Ethical Intervention Protocols

**Project:** Smart Student Retention & Early Warning System  
**Cutoff Horizon:** Day 35 ($t \le 35$)  
**Evaluation Focus:** Retrospective risk ranking and a proposed human-reviewed workflow ($K=500$ demonstration)  

---

## 1. Operational Framework

1. **Priority Triage Tiers:**
   - **Tier 1 (Urgent):** Proposed demonstration score band $\ge 0.70$ (candidate review target: 3 business days; not a validated SLA).
   - **Tier 2 (Watch):** Proposed demonstration score band $0.40 \le p < 0.70$ (candidate review target: 7 business days; not a validated SLA).
   - **Tier 3 (Standard):** Proposed demonstration score band $< 0.40$ (no response target established).
2. **Demonstration Capacity:** The retrospective queue is capped at up to $K=500$ enrollment records per presentation; this does not establish staffed advising capacity or unique-student caseload.
3. **Transparent Behavioral Signals:** Advisors receive concrete observational indicators (e.g., missed scheduled assessments, days since last login) rather than uninterpretable probability scores.

In [ ]:
import sys
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    current = Path.cwd().resolve()
    for directory in [current, *current.parents]:
        if (directory / 'pyproject.toml').is_file():
            return directory
    return current.parent

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from student_retention.decision_support.intervention_policy import (
    get_advising_workflow_principles,
)
from student_retention.decision_support.prioritize_students import (
    audit_support_queue_capacity,
    build_student_support_queue,
)
from student_retention.modeling.calibration import load_calibrated_model
from student_retention.modeling.predict import load_model
from student_retention.modeling.splits import split_by_student_group
from student_retention.paths import (
    MODELS_BASELINE_DIR,
    PREDICTIONS_DIR,
    PROCESSED_FEATURES_DIR,
)

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
# Load Feature Matrix & Calibrated Model
df = pd.read_parquet(PROCESSED_FEATURES_DIR / 'feature_matrix_policy_c.parquet')

# Isolate held-out test cohort to prevent in-sample ranking leakage
_, test_df = split_by_student_group(df, test_size=0.20, random_state=42, group_col='id_student')

# Load calibrated Random Forest model
cal_path = MODELS_BASELINE_DIR / 'random_forest_calibrated.joblib'
if cal_path.exists():
    model = load_calibrated_model(cal_path)
    print('Loaded calibrated Random Forest model artifact.')
else:
    model = load_model(MODELS_BASELINE_DIR / 'random_forest.joblib')
    print('Calibrated artifact not found; loaded baseline Random Forest.')

# Generate Student Support Queue on Held-Out Test Cohort with Capacity K=500 per presentation
queue_df = build_student_support_queue(
    feature_df=test_df,
    model=model,
    urgent_threshold=0.70,
    watch_threshold=0.40,
    capacity_k=500,
    save_path=PREDICTIONS_DIR / 'student_support_queue.csv',
)

# Audit capacity, unique students, and duplicate slots
cap_audit = audit_support_queue_capacity(queue_df)
print(f'\nGenerated demonstration queue on held-out test cohort with {len(queue_df):,} prioritized enrollment slots.')
print(f'Unique students in queue: {queue_df["student_key"].nunique():,}')
print(f'Duplicate slots occupied by multi-course students: {len(queue_df) - queue_df["student_key"].nunique():,}')

print('\n--- Caseload Capacity & Unique Student Audit ---')
print(cap_audit.to_string(index=False))

print('\n--- Caseload Breakdown by Priority Tier ---')
print(queue_df['priority_tier'].value_counts())

print('\n--- Sample Prioritized Records ---')
queue_df[['student_key', 'code_module', 'code_presentation', 'risk_probability', 'priority_tier', 'support_reason_summary', 'recommended_next_step']].head(10)

In [ ]:
# Display Core Advising Governance & Ethical Communication Principles
principles = get_advising_workflow_principles()
for key, principle in principles.items():
    print(f'\n[{key.upper()}]:')
    print(f'  {principle}')

## 2. Recommendations for Institutional Adoption

1. **Evaluation Point:** Day 35 is the retrospective model cutoff and a candidate point for testing a human-reviewed workflow; this analysis does not show that outreach then is effective or optimally timed.
2. **Operational Unit & Capacity Policy:** Capacity is currently bounded at 500 enrollment records per term. Because concurrent course registrations produce duplicate slots for multi-course learners, institutional governance must define an aggregation policy (e.g. max vs mean module risk) to establish a true 500-unique-student operational queue.
3. **Non-Stigmatizing Communication:** Outreach templates must focus on study support and learning barrier inquiry, strictly avoiding algorithmic failure disclosures.
4. **Candidate Service Targets:** A pilot could evaluate 3-business-day Tier 1 and 7-business-day Tier 2 advisor-review targets. These are proposed settings, not empirically validated service levels.
5. **Prospective Evaluation:** If piloted, advisors could log contact and outcome codes. Intervention effectiveness has not been established and requires prospective evaluation.